# Evaluation Stage 2 V2 - Anti-Forgetting

## Objectif
Évaluer le modèle Stage 2 V2 depuis le checkpoint-4250 (entraînement 98% complet).

## Contexte
- **Stage 2 V1** : 45% accuracy (catastrophic forgetting - CMD 95%, autres ~0%)
- **Stage 2 V2** : Stratégies anti-forgetting appliquées
  - LR 10x plus bas (2e-6 vs 1e-5)
  - Replay buffer 15% (300 samples Stage 1)
  - Dataset rééquilibré (~8k/classe)
  - LoRA étendu (r=32 vs r=8)

## Objectifs V2
| Classe | V1 | Objectif V2 |
|--------|-----|-------------|
| CMD | 95.1% | ≥80% |
| CBSD | 27.8% | ≥65% |
| CGM | 0.0% | ≥45% |
| CBB | 9.1% | ≥45% |
| Healthy | 0.0% | ≥55% |
| **Global** | **45%** | **≥65%** |

In [1]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================

import os
import json
import re
import random
import shutil
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from collections import Counter, defaultdict
from tqdm import tqdm

# IMPORTANT: Import unsloth AVANT transformers
from unsloth import FastVisionModel
from peft import PeftModel

from transformers import AutoProcessor

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch version: 2.9.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
GPU Memory: 33.67 GB


In [2]:
# ============================================================
# CELL 2: CONFIGURATION
# ============================================================

# Chemins
BASE_DIR = Path("/home/hounfodjidagba/learning/cassava/new_implementation")
CHECKPOINT_DIR = BASE_DIR / "checkpoints" / "stage2_v2_anti_forgetting"
CHECKPOINT_PATH = CHECKPOINT_DIR / "checkpoint-4250"
STAGE2_DATA_PATH = BASE_DIR / "data" / "stage2" / "stage2_vqa_dataset.json"

# Modèle
MODEL_NAME = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
TARGET_IMAGE_SIZE = (512, 512)

# Classes
CLASS_NAMES = ["CBB", "CBSD", "CGM", "CMD", "Healthy"]
CLASS_FULL_NAMES = {
    "CBB": "Bactériose du manioc (CBB)",
    "CBSD": "Striure brune du manioc (CBSD)",
    "CGM": "Acarien vert du manioc (CGM)",
    "CMD": "Mosaïque du manioc (CMD)",
    "Healthy": "Sain"
}

# Résultats V1 pour comparaison
V1_RESULTS = {
    "CMD": 95.1,
    "CBSD": 27.8,
    "CGM": 0.0,
    "CBB": 9.1,
    "Healthy": 0.0,
    "global": 45.0
}

print(f"{'='*60}")
print(f"CONFIGURATION EVALUATION STAGE 2 V2")
print(f"{'='*60}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Existe: {CHECKPOINT_PATH.exists()}")
print(f"{'='*60}")

CONFIGURATION EVALUATION STAGE 2 V2
Checkpoint: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v2_anti_forgetting/checkpoint-4250
Existe: True


In [3]:
# ============================================================
# CELL 3: CHARGEMENT MODÈLE DEPUIS CHECKPOINT
# ============================================================

print("\n" + "="*60)
print("LOADING MODEL FROM CHECKPOINT-4250")
print("="*60)

# Charger le modèle de base
print("\n1. Loading base model...")
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

# Charger les poids LoRA depuis le checkpoint
print("\n2. Loading LoRA weights from checkpoint...")
model = PeftModel.from_pretrained(model, str(CHECKPOINT_PATH))

# Charger le processor
print("\n3. Loading processor...")
processor = AutoProcessor.from_pretrained(str(CHECKPOINT_PATH))

# Mode inférence
print("\n4. Setting inference mode...")
FastVisionModel.for_inference(model)

print(f"\n✅ Model loaded from checkpoint-4250")
print(f"   Training was at step 4250/4534 (93.7% complete)")
print(f"   Final validation loss: ~0.810")
print("="*60)


LOADING MODEL FROM CHECKPOINT-4250

1. Loading base model...
==((====))==  Unsloth 2026.1.1: Fast Qwen2_5_Vl patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

2. Loading LoRA weights from checkpoint...

3. Loading processor...

4. Setting inference mode...

✅ Model loaded from checkpoint-4250
   Training was at step 4250/4534 (93.7% complete)
   Final validation loss: ~0.810


In [4]:
# ============================================================
# CELL 4: CHARGEMENT DONNÉES DE VALIDATION
# ============================================================

print("\n" + "="*60)
print("LOADING VALIDATION DATA")
print("="*60)

# Charger Stage 2 dataset
with open(STAGE2_DATA_PATH, 'r', encoding='utf-8') as f:
    stage2_data = json.load(f)

print(f"Total Stage 2 samples: {len(stage2_data)}")

def extract_image_path(text):
    """Extrait le chemin de l'image."""
    match = re.search(r'<img>(.*?)</img>', text)
    return match.group(1) if match else None

def format_sample(sample):
    """Formate un sample pour l'évaluation."""
    conversations = sample['conversations']
    image_path = None
    question = ""
    expected_answer = ""
    
    for turn in conversations:
        if turn['from'] == 'user':
            image_path = extract_image_path(turn['value'])
            question = re.sub(r'Picture \d+: <img>.*?</img>\n?', '', turn['value']).strip()
        elif turn['from'] == 'assistant':
            expected_answer = turn['value']
    
    return {
        'id': sample['id'],
        'image_path': image_path,
        'question': question,
        'expected_answer': expected_answer,
        'class_name': sample['metadata']['class'],
        'qa_type': sample['metadata'].get('qa_type', 'unknown'),
        'severity': sample['metadata'].get('severity', 'unknown')
    }

# Formater tous les samples
all_samples = [format_sample(s) for s in stage2_data]

# Filtrer les images valides
valid_samples = [s for s in all_samples if s['image_path'] and os.path.exists(s['image_path'])]
print(f"Valid samples: {len(valid_samples)}")

# Créer un set de validation (10%)
random.seed(42)
random.shuffle(valid_samples)
val_samples = valid_samples[:int(len(valid_samples) * 0.1)]

print(f"Validation samples: {len(val_samples)}")

# Distribution par classe
val_dist = Counter(s['class_name'] for s in val_samples)
print("\nDistribution validation:")
for cls in CLASS_NAMES:
    print(f"  {cls}: {val_dist[cls]}")


LOADING VALIDATION DATA
Total Stage 2 samples: 29276
Valid samples: 29276
Validation samples: 2927

Distribution validation:
  CBB: 379
  CBSD: 550
  CGM: 489
  CMD: 1205
  Healthy: 304


In [5]:
# ============================================================
# CELL 5: FONCTIONS D'ÉVALUATION
# ============================================================

def generate_response(image_path: str, question: str, max_new_tokens: int = 256) -> str:
    """Génère une réponse pour une image et une question."""
    try:
        image = Image.open(image_path).convert('RGB').resize(TARGET_IMAGE_SIZE)
    except Exception as e:
        return f"Erreur image: {e}"
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question}
            ]
        }
    ]
    
    input_text = processor.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = processor(
        text=[input_text],
        images=[[image]],
        return_tensors="pt",
        padding=True,
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=processor.tokenizer.pad_token_id,
        )
    
    response = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extraire seulement la réponse de l'assistant
    if "assistant" in response.lower():
        parts = response.lower().split("assistant")
        response = parts[-1].strip()
    
    return response

def extract_disease(response: str) -> str:
    """Extrait la classe de maladie de la réponse."""
    response_lower = response.lower()
    
    # Patterns pour chaque classe
    if "cmd" in response_lower or "mosaïque" in response_lower or "mosaic" in response_lower:
        return "CMD"
    elif "cbsd" in response_lower or "striure" in response_lower or "brown streak" in response_lower:
        return "CBSD"
    elif "cbb" in response_lower or "bactériose" in response_lower or "bacterial" in response_lower:
        return "CBB"
    elif "cgm" in response_lower or "acarien" in response_lower or "green mite" in response_lower or "mottle" in response_lower:
        return "CGM"
    elif "sain" in response_lower or "healthy" in response_lower or "aucun" in response_lower:
        return "Healthy"
    
    return "Unknown"

print("✅ Fonctions d'évaluation définies")

✅ Fonctions d'évaluation définies


In [6]:
# ============================================================
# CELL 6: ÉVALUATION CLASSIFICATION
# ============================================================

print("\n" + "="*60)
print("ÉVALUATION CLASSIFICATION")
print("="*60)

# Sélectionner des samples pour l'évaluation de classification
# Prendre des samples équilibrés par classe
N_PER_CLASS = 40  # 40 samples par classe = 200 total

samples_by_class = defaultdict(list)
for s in val_samples:
    samples_by_class[s['class_name']].append(s)

eval_samples = []
for cls in CLASS_NAMES:
    cls_samples = samples_by_class[cls]
    n_take = min(N_PER_CLASS, len(cls_samples))
    eval_samples.extend(random.sample(cls_samples, n_take))

random.shuffle(eval_samples)
print(f"\nÉvaluation sur {len(eval_samples)} samples")

# Question de classification
classification_question = "Quelle maladie affecte cette feuille de manioc ?"

# Évaluer
results = []
for sample in tqdm(eval_samples, desc="Evaluating"):
    response = generate_response(sample['image_path'], classification_question)
    predicted = extract_disease(response)
    actual = sample['class_name']
    
    results.append({
        'id': sample['id'],
        'actual': actual,
        'predicted': predicted,
        'correct': predicted == actual,
        'response': response[:200]  # Tronquer pour le stockage
    })

# Calculer les métriques
correct = sum(1 for r in results if r['correct'])
total = len(results)
global_accuracy = correct / total * 100

print(f"\n" + "="*60)
print(f"RÉSULTATS CLASSIFICATION")
print(f"="*60)
print(f"\n📊 Accuracy globale: {global_accuracy:.1f}% ({correct}/{total})")


ÉVALUATION CLASSIFICATION

Évaluation sur 200 samples


Evaluating: 100%|██████████| 200/200 [08:18<00:00,  2.49s/it]


RÉSULTATS CLASSIFICATION

📊 Accuracy globale: 31.0% (62/200)


In [7]:
# ============================================================
# CELL 7: ANALYSE PAR CLASSE
# ============================================================

print("\n" + "="*60)
print("ANALYSE PAR CLASSE")
print("="*60)

# Métriques par classe
class_metrics = {}
for cls in CLASS_NAMES:
    cls_results = [r for r in results if r['actual'] == cls]
    if cls_results:
        cls_correct = sum(1 for r in cls_results if r['correct'])
        cls_total = len(cls_results)
        cls_accuracy = cls_correct / cls_total * 100
        class_metrics[cls] = {
            'accuracy': cls_accuracy,
            'correct': cls_correct,
            'total': cls_total
        }

print("\n📈 Accuracy par classe (V2 vs V1):")
print("-" * 70)
print(f"{'Classe':<30} {'V2':>10} {'V1':>10} {'Δ':>10} {'Status':>8}")
print("-" * 70)

for cls in CLASS_NAMES:
    if cls in class_metrics:
        v2_acc = class_metrics[cls]['accuracy']
        v1_acc = V1_RESULTS.get(cls, 0)
        diff = v2_acc - v1_acc
        
        if diff > 5:
            status = "✅ MIEUX"
        elif diff > -5:
            status = "➖ STABLE"
        else:
            status = "❌ PIRE"
        
        print(f"{CLASS_FULL_NAMES[cls]:<30} {v2_acc:>9.1f}% {v1_acc:>9.1f}% {diff:>+9.1f}% {status:>8}")

print("-" * 70)
v1_global = V1_RESULTS['global']
diff_global = global_accuracy - v1_global
status_global = "✅" if diff_global > 0 else "❌"
print(f"{'GLOBAL':<30} {global_accuracy:>9.1f}% {v1_global:>9.1f}% {diff_global:>+9.1f}% {status_global:>8}")
print("-" * 70)


ANALYSE PAR CLASSE

📈 Accuracy par classe (V2 vs V1):
----------------------------------------------------------------------
Classe                                 V2         V1          Δ   Status
----------------------------------------------------------------------
Bactériose du manioc (CBB)          40.0%       9.1%     +30.9%  ✅ MIEUX
Striure brune du manioc (CBSD)       5.0%      27.8%     -22.8%   ❌ PIRE
Acarien vert du manioc (CGM)        27.5%       0.0%     +27.5%  ✅ MIEUX
Mosaïque du manioc (CMD)            75.0%      95.1%     -20.1%   ❌ PIRE
Sain                                 7.5%       0.0%      +7.5%  ✅ MIEUX
----------------------------------------------------------------------
GLOBAL                              31.0%      45.0%     -14.0%        ❌
----------------------------------------------------------------------


In [8]:
# ============================================================
# CELL 8: MATRICE DE CONFUSION
# ============================================================

print("\n" + "="*60)
print("MATRICE DE CONFUSION")
print("="*60)

# Créer la matrice
confusion = {actual: {pred: 0 for pred in CLASS_NAMES + ['Unknown']} for actual in CLASS_NAMES}

for r in results:
    actual = r['actual']
    pred = r['predicted']
    if actual in confusion:
        confusion[actual][pred] += 1

# Afficher la matrice
header = f"{'Actual/Pred':<12}" + "".join(f"{c:>10}" for c in CLASS_NAMES + ['Unknown'])
print(f"\n{header}")
print("-" * len(header))

for actual in CLASS_NAMES:
    row = f"{actual:<12}"
    for pred in CLASS_NAMES + ['Unknown']:
        count = confusion[actual][pred]
        if count > 0:
            row += f"{count:>10}"
        else:
            row += f"{'·':>10}"
    print(row)

# Analyser les confusions principales
print("\n📊 Confusions principales:")
confusions = []
for actual in CLASS_NAMES:
    for pred in CLASS_NAMES + ['Unknown']:
        if actual != pred and confusion[actual][pred] > 0:
            confusions.append((actual, pred, confusion[actual][pred]))

confusions.sort(key=lambda x: x[2], reverse=True)
for actual, pred, count in confusions[:10]:
    print(f"  {actual} → {pred}: {count} erreurs")


MATRICE DE CONFUSION

Actual/Pred        CBB      CBSD       CGM       CMD   Healthy   Unknown
------------------------------------------------------------------------
CBB                 16         3         7        13         1         ·
CBSD                 7         2         6        22         3         ·
CGM                  1         4        11        24         ·         ·
CMD                  2         4         4        30         ·         ·
Healthy             15         3        11         8         3         ·

📊 Confusions principales:
  CGM → CMD: 24 erreurs
  CBSD → CMD: 22 erreurs
  Healthy → CBB: 15 erreurs
  CBB → CMD: 13 erreurs
  Healthy → CGM: 11 erreurs
  Healthy → CMD: 8 erreurs
  CBB → CGM: 7 erreurs
  CBSD → CBB: 7 erreurs
  CBSD → CGM: 6 erreurs
  CGM → CBSD: 4 erreurs


In [9]:
# ============================================================
# CELL 9: TEST VQA (QUESTIONS VARIÉES)
# ============================================================

print("\n" + "="*60)
print("TEST VQA - QUESTIONS VARIÉES")
print("="*60)

# Sélectionner quelques samples pour tester différentes questions
test_samples = random.sample(eval_samples, min(5, len(eval_samples)))

vqa_questions = [
    "Quelle maladie affecte cette feuille de manioc ?",
    "Décris les symptômes visibles sur cette feuille.",
    "Quelle est la sévérité de l'infection ?",
    "Quels traitements recommandes-tu pour cette maladie ?"
]

for i, sample in enumerate(test_samples):
    print(f"\n{'='*60}")
    print(f"Sample {i+1}: {sample['class_name']}")
    print(f"Image: {Path(sample['image_path']).name}")
    print(f"{'='*60}")
    
    for q in vqa_questions:
        response = generate_response(sample['image_path'], q, max_new_tokens=150)
        print(f"\n❓ {q}")
        print(f"💬 {response[:300]}..." if len(response) > 300 else f"💬 {response}")


TEST VQA - QUESTIONS VARIÉES

Sample 1: Healthy
Image: 1953482304.jpg

❓ Quelle maladie affecte cette feuille de manioc ?
💬 cette feuille présente des symptômes de **acarien vert du manioc (cgm)**. niveau de sévérité : dégâts d'acariens. les signes visibles incluent : taches chlorotiques angulaires, limitées par les venons, couleur vert foncé, marron foncé en profondeur, motifs jaune-vert puis jaune sur les feuilles matu...

❓ Décris les symptômes visibles sur cette feuille.
💬 cette image montre : taches angulaires jaunes/couleur de papier d'épicerie, aspect moucheté sur face supérieure des feuilles, flétrissement sévère possible en fin de saison. ces signes sont typiques de la bactériose du manioc au stade infection bactérienne.

❓ Quelle est la sévérité de l'infection ?
💬 cette feuille de manioc est **saine**. elle présente les caractéristiques normales d'un plant en bonne santé : couleur vert foncé à vert moyen uniforme, consistance ferme et lisse, tiges sans lésions ni exsudat. co

In [10]:
# ============================================================
# CELL 10: SAUVEGARDE RÉSULTATS ET COPIE FINAL_MODEL
# ============================================================

print("\n" + "="*60)
print("SAUVEGARDE DES RÉSULTATS")
print("="*60)

# Préparer les résultats
eval_results = {
    "stage": "2_v2_anti_forgetting",
    "checkpoint": "checkpoint-4250",
    "training_completion": "93.7% (4250/4534 steps)",
    "final_loss": 0.810,
    
    "evaluation": {
        "n_samples": len(results),
        "global_accuracy_v2": global_accuracy,
        "global_accuracy_v1": V1_RESULTS['global'],
        "improvement": global_accuracy - V1_RESULTS['global'],
    },
    
    "class_metrics_v2": {cls: class_metrics[cls]['accuracy'] for cls in class_metrics},
    "class_metrics_v1": {cls: V1_RESULTS[cls] for cls in CLASS_NAMES},
    
    "confusion_summary": [
        {"actual": a, "predicted": p, "count": c} 
        for a, p, c in confusions[:10]
    ],
    
    "anti_forgetting_config": {
        "learning_rate": "2e-6 (10x lower than V1)",
        "replay_buffer": "300 samples (15%)",
        "lora_rank": 32,
        "warmup_ratio": 0.15,
        "target_per_class": 8000
    }
}

# Sauvegarder
results_path = CHECKPOINT_DIR / "eval_results_v2.json"
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(eval_results, f, indent=2, ensure_ascii=False)

print(f"✅ Résultats sauvegardés: {results_path}")

# Copier checkpoint comme final_model
final_model_path = CHECKPOINT_DIR / "final_model"
if not final_model_path.exists():
    print(f"\nCopie du checkpoint vers final_model...")
    shutil.copytree(CHECKPOINT_PATH, final_model_path)
    print(f"✅ Checkpoint copié vers: {final_model_path}")
else:
    print(f"\n⚠️ final_model existe déjà: {final_model_path}")


SAUVEGARDE DES RÉSULTATS
✅ Résultats sauvegardés: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v2_anti_forgetting/eval_results_v2.json

Copie du checkpoint vers final_model...
✅ Checkpoint copié vers: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v2_anti_forgetting/final_model


In [11]:
# ============================================================
# CELL 11: CONCLUSION
# ============================================================

print("\n" + "="*60)
print("CONCLUSION - STAGE 2 V2 vs V1")
print("="*60)

improvement = global_accuracy - V1_RESULTS['global']

print(f"\n📊 RÉSUMÉ:")
print(f"   - Accuracy V1: {V1_RESULTS['global']:.1f}%")
print(f"   - Accuracy V2: {global_accuracy:.1f}%")
print(f"   - Amélioration: {improvement:+.1f}%")

print(f"\n📈 OBJECTIFS ATTEINTS:")
objectives = {
    "CMD": 80, "CBSD": 65, "CGM": 45, "CBB": 45, "Healthy": 55, "Global": 65
}

for cls in CLASS_NAMES:
    if cls in class_metrics:
        v2_acc = class_metrics[cls]['accuracy']
        target = objectives[cls]
        status = "✅" if v2_acc >= target else "❌"
        print(f"   {status} {cls}: {v2_acc:.1f}% (objectif: ≥{target}%)")

global_target = objectives['Global']
global_status = "✅" if global_accuracy >= global_target else "❌"
print(f"   {global_status} Global: {global_accuracy:.1f}% (objectif: ≥{global_target}%)")

if improvement > 0:
    print(f"\n🎉 Les stratégies anti-forgetting ont fonctionné!")
    print(f"   Le modèle V2 est meilleur de {improvement:.1f}% par rapport à V1.")
else:
    print(f"\n⚠️ Les stratégies anti-forgetting n'ont pas suffisamment amélioré le modèle.")
    print(f"   Considérer d'autres approches (EWC, plus de replay, etc.)")

print(f"\n📁 Fichiers générés:")
print(f"   - {results_path}")
print(f"   - {final_model_path}")
print("\n" + "="*60)


CONCLUSION - STAGE 2 V2 vs V1

📊 RÉSUMÉ:
   - Accuracy V1: 45.0%
   - Accuracy V2: 31.0%
   - Amélioration: -14.0%

📈 OBJECTIFS ATTEINTS:
   ❌ CBB: 40.0% (objectif: ≥45%)
   ❌ CBSD: 5.0% (objectif: ≥65%)
   ❌ CGM: 27.5% (objectif: ≥45%)
   ❌ CMD: 75.0% (objectif: ≥80%)
   ❌ Healthy: 7.5% (objectif: ≥55%)
   ❌ Global: 31.0% (objectif: ≥65%)

⚠️ Les stratégies anti-forgetting n'ont pas suffisamment amélioré le modèle.
   Considérer d'autres approches (EWC, plus de replay, etc.)

📁 Fichiers générés:
   - /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v2_anti_forgetting/eval_results_v2.json
   - /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v2_anti_forgetting/final_model

